In [1]:
# Cell 1: Setup
import os, gc, json, time
from pathlib import Path
import numpy as np, pandas as pd
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as exc:
    print('Không mount được Drive:', exc)

QUICK = False                     # True: only the first QUICK_ROWS rows of each file
QUICK_ROWS = 200_000
CHUNK = 1_000_000
ROOT = Path('/content/drive/MyDrive')
OUT = ROOT / 'THAT_IDS_RERUN' / 'journal_step19_dataset_audit_v1' / ('quick' if QUICK else 'paper')
OUT.mkdir(parents=True, exist_ok=True)
DATASETS = {
    'UNSW_V2': {'path': ROOT / 'NF-UNSW-NB15-v2.csv', 'pair_of': None},
    'UNSW_V3': {'path': ROOT / 'NF-UNSW-NB15-v3.csv', 'pair_of': 'UNSW_V2'},
    'CIC_V2':  {'path': ROOT / 'NF-CSE-CIC-IDS2018-v2.csv', 'pair_of': None},
    'CIC_V3':  {'path': ROOT / 'NF-CSE-CIC-IDS2018-v3.csv', 'pair_of': 'CIC_V2'},
}
EXPECTED = {'CIC_V2': {'unique_vectors': 8_981_062, 'unique_host_pairs': 1_866_048},
            'UNSW_V2': {'unique_vectors': 236_999, 'unique_host_pairs': 153}}
ID_COLS = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT']
LABEL_COL, ATTACK_COL = 'Label', 'Attack'
SPLIT_SEEDS = [42, 123, 456]
TRAIN_FRAC, VAL_FRAC = 0.70, 0.10
for k, v in DATASETS.items():
    print('%-8s %s %s' % (k, 'OK ' if v['path'].exists() else 'MISSING', v['path']))

Mounted at /content/drive
UNSW_V2  OK  /content/drive/MyDrive/NF-UNSW-NB15-v2.csv
UNSW_V3  OK  /content/drive/MyDrive/NF-UNSW-NB15-v3.csv
CIC_V2   OK  /content/drive/MyDrive/NF-CSE-CIC-IDS2018-v2.csv
CIC_V3   OK  /content/drive/MyDrive/NF-CSE-CIC-IDS2018-v3.csv


In [2]:
# Cell 2: Columns: model features, absolute timestamps, v3-only temporal features
headers = {k: list(pd.read_csv(v['path'], nrows=0).columns) for k, v in DATASETS.items()}
def is_timestamp(c):
    u = c.upper()
    return ('FLOW_START' in u) or ('FLOW_END' in u)

COLS = {}
for k, h in headers.items():
    ts = [c for c in h if is_timestamp(c)]
    extra = []
    if DATASETS[k]['pair_of']:
        extra = [c for c in h if c not in headers[DATASETS[k]['pair_of']] and c not in ts]
    drop = set(ID_COLS + [LABEL_COL, ATTACK_COL, 'Dataset'] + ts)
    feats_a = [c for c in h if c not in drop]
    feats_b = [c for c in feats_a if c not in extra]
    COLS[k] = {'timestamps': ts, 'v3_only': extra, 'features_a': feats_a, 'features_b': feats_b}
    print('\n%s: %d cột | %d đặc trưng (a) | %d đặc trưng (b)' % (k, len(h), len(feats_a), len(feats_b)))
    print('  timestamp tuyệt đối:', ts)
    print('  chỉ có ở v3 (bỏ trong biến thể b):', extra)
    missing = [c for c in ID_COLS + [LABEL_COL, ATTACK_COL] if c not in h]
    if missing:
        raise RuntimeError('%s thiếu cột %s' % (k, missing))
for k in ['UNSW_V3', 'CIC_V3']:
    if len(COLS[k]['v3_only']) != 10:
        print('[CẢNH BÁO] %s có %d đặc trưng chỉ có ở v3 (kỳ vọng 10). Kiểm tra danh sách trên.' % (k, len(COLS[k]['v3_only'])))
    if not COLS[k]['timestamps']:
        print('[CẢNH BÁO] %s không tìm thấy cột timestamp.' % k)
atomic = lambda o, p: (open(str(p) + '.tmp', 'w').write(json.dumps(o, indent=1, ensure_ascii=False)), os.replace(str(p) + '.tmp', p))
atomic(COLS, OUT / 'columns.json')


UNSW_V2: 45 cột | 39 đặc trưng (a) | 39 đặc trưng (b)
  timestamp tuyệt đối: []
  chỉ có ở v3 (bỏ trong biến thể b): []

UNSW_V3: 55 cột | 47 đặc trưng (a) | 39 đặc trưng (b)
  timestamp tuyệt đối: ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS']
  chỉ có ở v3 (bỏ trong biến thể b): ['SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV', 'DST_TO_SRC_IAT_MIN', 'DST_TO_SRC_IAT_MAX', 'DST_TO_SRC_IAT_AVG', 'DST_TO_SRC_IAT_STDDEV']

CIC_V2: 45 cột | 39 đặc trưng (a) | 39 đặc trưng (b)
  timestamp tuyệt đối: []
  chỉ có ở v3 (bỏ trong biến thể b): []

CIC_V3: 55 cột | 47 đặc trưng (a) | 39 đặc trưng (b)
  timestamp tuyệt đối: ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS']
  chỉ có ở v3 (bỏ trong biến thể b): ['SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV', 'DST_TO_SRC_IAT_MIN', 'DST_TO_SRC_IAT_MAX', 'DST_TO_SRC_IAT_AVG', 'DST_TO_SRC_IAT_STDDEV']
[CẢNH BÁO] UNSW_V3 có 8 đặc trưng chỉ có ở v3 (kỳ vọng 10)

(8612, None)

In [3]:
# Cell 3: One streaming pass per dataset: hashes, host pairs, families, timestamps
def vec_hash(df, cols):
    x = df[cols].apply(pd.to_numeric, errors='coerce').astype(np.float64)
    x = x.replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)
    return pd.util.hash_pandas_object(x, index=False).to_numpy(np.uint64)

def pair_hash(df):
    s = df['IPV4_SRC_ADDR'].fillna('<MISSING_IP>').astype(str).to_numpy()
    d = df['IPV4_DST_ADDR'].fillna('<MISSING_IP>').astype(str).to_numpy()
    left = np.where(s <= d, s, d); right = np.where(s <= d, d, s)
    return pd.util.hash_pandas_object(pd.DataFrame({'l': left, 'r': right}), index=False).to_numpy(np.uint64)

def ip_hash(series):
    return pd.util.hash_pandas_object(series.fillna('<MISSING_IP>').astype(str), index=False).to_numpy(np.uint64)

def scan(name):
    cfg, cols = DATASETS[name], COLS[name]
    usecols = list(dict.fromkeys(ID_COLS + [LABEL_COL, ATTACK_COL] + cols['timestamps'] + cols['features_a']))
    parts = {k: [] for k in ['ha', 'hb', 'pair', 'src', 'dst', 'label', 'attack', 'start']}
    t0 = time.time(); n = 0
    reader = pd.read_csv(cfg['path'], usecols=usecols, chunksize=CHUNK, low_memory=False,
                         nrows=QUICK_ROWS if QUICK else None)
    for df in reader:
        parts['ha'].append(vec_hash(df, cols['features_a']))
        parts['hb'].append(vec_hash(df, cols['features_b']) if cols['features_b'] != cols['features_a'] else None)
        parts['pair'].append(pair_hash(df))
        parts['src'].append(ip_hash(df['IPV4_SRC_ADDR'])); parts['dst'].append(ip_hash(df['IPV4_DST_ADDR']))
        parts['label'].append(pd.to_numeric(df[LABEL_COL], errors='coerce').fillna(0).astype(np.int8).to_numpy())
        parts['attack'].append(df[ATTACK_COL].astype(str).str.strip().to_numpy())
        if cols['timestamps']:
            st = [c for c in cols['timestamps'] if 'START' in c.upper()][0]
            parts['start'].append(pd.to_numeric(df[st], errors='coerce').to_numpy(np.float64))
        n += len(df)
        print('  %s: %s dòng (%.1f phút)' % (name, f'{n:,}', (time.time() - t0) / 60))
        del df; gc.collect()
    out = {}
    for k, v in parts.items():
        v = [a for a in v if a is not None]
        out[k] = np.concatenate(v) if v else None
    fam_codes, fam_names = pd.factorize(pd.Series(out['attack']), sort=True)
    out['family'] = fam_codes.astype(np.int16); out['family_names'] = [str(x) for x in fam_names]
    del out['attack']; gc.collect()
    return out

DATA = {}
for name in DATASETS:
    print('\n>>> ' + name)
    DATA[name] = scan(name)


>>> UNSW_V2
  UNSW_V2: 1,000,000 dòng (0.1 phút)
  UNSW_V2: 2,000,000 dòng (0.2 phút)
  UNSW_V2: 2,390,275 dòng (0.2 phút)

>>> UNSW_V3
  UNSW_V3: 1,000,000 dòng (0.2 phút)
  UNSW_V3: 2,000,000 dòng (0.3 phút)
  UNSW_V3: 2,365,424 dòng (0.4 phút)

>>> CIC_V2


/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


  CIC_V2: 1,000,000 dòng (0.1 phút)


/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


  CIC_V2: 2,000,000 dòng (0.3 phút)


/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


... (29 outputs left out)

/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


  CIC_V2: 18,000,000 dòng (2.0 phút)


/usr/local/lib/python3.13/dist-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


  CIC_V2: 18,893,708 dòng (2.1 phút)

>>> CIC_V3
  CIC_V3: 1,000,000 dòng (0.2 phút)
  CIC_V3: 2,000,000 dòng (0.3 phút)
  CIC_V3: 3,000,000 dòng (0.4 phút)
  CIC_V3: 4,000,000 dòng (0.5 phút)
  CIC_V3: 5,000,000 dòng (0.6 phút)
  CIC_V3: 6,000,000 dòng (0.7 phút)
  CIC_V3: 7,000,000 dòng (0.8 phút)
  CIC_V3: 8,000,000 dòng (0.9 phút)
  CIC_V3: 9,000,000 dòng (1.0 phút)
  CIC_V3: 10,000,000 dòng (1.2 phút)
  CIC_V3: 11,000,000 dòng (1.3 phút)
  CIC_V3: 12,000,000 dòng (1.4 phút)
  CIC_V3: 13,000,000 dòng (1.6 phút)
  CIC_V3: 14,000,000 dòng (1.7 phút)
  CIC_V3: 15,000,000 dòng (1.8 phút)
  CIC_V3: 16,000,000 dòng (2.0 phút)
  CIC_V3: 17,000,000 dòng (2.1 phút)
  CIC_V3: 18,000,000 dòng (2.2 phút)
  CIC_V3: 19,000,000 dòng (2.4 phút)
  CIC_V3: 20,000,000 dòng (2.5 phút)
  CIC_V3: 20,115,529 dòng (2.5 phút)


In [4]:
# Cell 4: Basic statistics, duplicates and host-pair concentration per family
def dup_stats(h):
    u, c = np.unique(h, return_counts=True)
    return {'unique_vectors': int(len(u)), 'repeated_rows_beyond_first': int(len(h) - len(u)),
            'repeated_rows_rate': float((len(h) - len(u)) / len(h)),
            'rows_in_duplicate_groups_rate': float(c[c > 1].sum() / len(h))}

summary, fam_tables = {}, []
for name, d in DATA.items():
    y, fam, names = d['label'], d['family'], d['family_names']
    benign_code = int(np.argmax([((fam == i) & (y == 0)).sum() for i in range(len(names))]))
    s = {'rows': int(len(y)), 'attack_rate': float(y.mean()), 'families': len(names) - 1,
         'benign_family_name': names[benign_code],
         'dup_a': dup_stats(d['ha']), 'dup_b': dup_stats(d['hb']) if d['hb'] is not None else None,
         'unique_ips': int(len(np.unique(np.concatenate([d['src'], d['dst']])))),
         'unique_host_pairs': int(len(np.unique(d['pair'])))}
    if d['start'] is not None:
        st = d['start'][np.isfinite(d['start'])]
        s['start_min'], s['start_max'] = float(st.min()), float(st.max())
        s['rows_without_start'] = int((~np.isfinite(d['start'])).sum())
    if name in EXPECTED and not QUICK:
        for key, val in EXPECTED[name].items():
            got = s['dup_a']['unique_vectors'] if key == 'unique_vectors' else s[key]
            s['check_' + key] = 'OK' if got == val else 'MISMATCH (got %d, expected %d)' % (got, val)
    summary[name] = s
    for i, fn in enumerate(names):
        m = fam == i
        pairs, cnt = np.unique(d['pair'][m], return_counts=True)
        fam_tables.append({'dataset': name, 'family': fn, 'is_benign': i == benign_code, 'rows': int(m.sum()),
                           'host_pairs': int(len(pairs)), 'top_pair_share': float(cnt.max() / m.sum()) if m.any() else 0.0,
                           'pairs_for_90pct': int(np.searchsorted(np.cumsum(np.sort(cnt)[::-1]) / m.sum(), 0.90) + 1) if m.any() else 0})
fam_df = pd.DataFrame(fam_tables)
fam_df.to_csv(OUT / 'family_host_pairs.csv', index=False)
atomic(summary, OUT / 'summary.json')
print(json.dumps(summary, indent=1, ensure_ascii=False))
display(fam_df[~fam_df.is_benign].sort_values(['dataset', 'host_pairs']))

{
 "UNSW_V2": {
  "rows": 2390275,
  "attack_rate": 0.039766554057587514,
  "families": 9,
  "benign_family_name": "Benign",
  "dup_a": {
   "unique_vectors": 236999,
   "repeated_rows_beyond_first": 2153276,
   "repeated_rows_rate": 0.900848647122193,
   "rows_in_duplicate_groups_rate": 0.9472315946909874
  },
  "dup_b": null,
  "unique_ips": 44,
  "unique_host_pairs": 153,
  "check_unique_vectors": "OK",
  "check_unique_host_pairs": "OK"
 },
 "UNSW_V3": {
  "rows": 2365424,
  "attack_rate": 0.053983133679205084,
  "families": 9,
  "benign_family_name": "Benign",
  "dup_a": {
   "unique_vectors": 1230871,
   "repeated_rows_beyond_first": 1134553,
   "repeated_rows_rate": 0.4796404365559832,
   "rows_in_duplicate_groups_rate": 0.5598746778590223
  },
  "dup_b": {
   "unique_vectors": 889457,
   "repeated_rows_beyond_first": 1475967,
   "repeated_rows_rate": 0.6239756593321113,
   "rows_in_duplicate_groups_rate": 0.7168875432057846
  },
  "unique_ips": 44,
  "unique_host_pairs": 157,
  

,dataset,family,is_benign,rows,host_pairs,top_pair_share,pairs_for_90pct
27,CIC_V2,DoS attacks-GoldenEye,False,27723,1,1.000000,1
28,CIC_V2,DoS attacks-Hulk,False,432648,1,1.000000,1
29,CIC_V2,DoS attacks-SlowHTTPTest,False,14116,1,1.000000,1
30,CIC_V2,DoS attacks-Slowloris,False,9512,1,1.000000,1
31,CIC_V2,FTP-BruteForce,False,25933,1,1.000000,1
33,CIC_V2,SQL Injection,False,432,1,1.000000,1
34,CIC_V2,SSH-Bruteforce,False,94979,1,1.000000,1
23,CIC_V2,Brute Force -XSS,False,927,4,0.949299,1
22,CIC_V2,Brute Force -Web,False,2143,7,0.969202,1
21,CIC_V2,Bot,False,143097,10,0.111058,9


In [5]:
# Cell 5: Simulated host-pair splits (same algorithm as step11) and v3 temporal splits
def host_pair_assignment(pair, seed):
    u, inv, cnt = np.unique(pair, return_inverse=True, return_counts=True)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(u)); cum = np.cumsum(cnt[order])
    t_end = int(np.searchsorted(cum, int(len(pair) * TRAIN_FRAC), side='left') + 1)
    v_end = int(np.searchsorted(cum, int(len(pair) * (TRAIN_FRAC + VAL_FRAC)), side='left') + 1)
    t_end = min(max(t_end, 1), len(order) - 2); v_end = min(max(v_end, t_end + 1), len(order) - 1)
    g = np.full(len(u), 2, np.int8); g[order[:t_end]] = 0; g[order[t_end:v_end]] = 1
    return g[inv], {'train_groups': t_end, 'val_groups': v_end - t_end, 'test_groups': len(u) - v_end}

def temporal_assignment(start):
    order = np.argsort(np.where(np.isfinite(start), start, np.inf), kind='mergesort')
    a = np.empty(len(start), np.int8); n = len(start)
    a[order[:int(n * TRAIN_FRAC)]] = 0; a[order[int(n * TRAIN_FRAC):int(n * (TRAIN_FRAC + VAL_FRAC))]] = 1
    a[order[int(n * (TRAIN_FRAC + VAL_FRAC)):]] = 2
    return a

def describe(name, protocol, seed, assign, extra=None):
    d = DATA[name]; y, fam, names = d['label'], d['family'], d['family_names']
    benign = summary[name]['benign_family_name']
    row = {'dataset': name, 'protocol': protocol, 'seed': seed}
    for s, lab in enumerate(['train', 'val', 'test']):
        m = assign == s
        row[lab + '_rows'] = int(m.sum()); row[lab + '_attack_rate'] = float(y[m].mean()) if m.any() else float('nan')
    present = {lab: set(np.unique(fam[(assign == s) & (y == 1)])) for s, lab in enumerate(['train', 'val', 'test'])}
    attack_fams = [i for i, n in enumerate(names) if n != benign]
    for lab in ['train', 'val', 'test']:
        row['absent_from_' + lab] = ', '.join(names[i] for i in attack_fams if i not in present[lab])
    te_att = (assign == 2) & (y == 1)
    unseen = te_att & ~np.isin(fam, list(present['train']))
    row['test_attack_unseen_in_train_share'] = float(unseen.sum() / max(te_att.sum(), 1))
    row['test_hash_a_seen_in_train_rate'] = float(np.isin(d['ha'][assign == 2], np.unique(d['ha'][assign == 0])).mean())
    if extra:
        row.update(extra)
    return row

rows = []
for name in DATA:
    for seed in SPLIT_SEEDS:
        a, groups = host_pair_assignment(DATA[name]['pair'], seed)
        rows.append(describe(name, 'host_pair', seed, a, groups))
    if DATA[name]['start'] is not None:
        rows.append(describe(name, 'temporal', None, temporal_assignment(DATA[name]['start'])))
splits = pd.DataFrame(rows)
splits.to_csv(OUT / 'split_simulation.csv', index=False)
pd.set_option('display.max_colwidth', 120)
display(splits[['dataset', 'protocol', 'seed', 'test_attack_rate', 'test_attack_unseen_in_train_share',
                'absent_from_train', 'absent_from_test', 'test_hash_a_seen_in_train_rate']])

,dataset,protocol,seed,test_attack_rate,test_attack_unseen_in_train_share,absent_from_train,absent_from_test,test_hash_a_seen_in_train_rate
0,UNSW_V2,host_pair,42.0,0.020000,0.000000,,,0.939156
1,UNSW_V2,host_pair,123.0,0.031578,0.000000,,,0.940443
2,UNSW_V2,host_pair,456.0,0.052517,0.000000,,,0.934780
3,UNSW_V3,host_pair,42.0,0.033912,0.000000,,,0.527211
4,UNSW_V3,host_pair,123.0,0.082248,0.000000,,,0.548335
5,UNSW_V3,host_pair,456.0,0.036845,0.000000,,,0.553124
6,UNSW_V3,temporal,NaN,0.087494,0.000000,,,0.516909
7,CIC_V2,host_pair,42.0,0.206648,0.587269,"DoS attacks-Hulk, DoS attacks-Slowloris, FTP-BruteForce","Brute Force -Web, Brute Force -XSS, DoS attacks-GoldenEye, DoS attacks-SlowHTTPTest, DoS attacks-Slowloris, SQL Inje...",0.478535
8,CIC_V2,host_pair,123.0,0.059532,0.123557,"DoS attacks-GoldenEye, SQL Injection","DoS attacks-Hulk, DoS attacks-SlowHTTPTest, DoS attacks-Slowloris, FTP-BruteForce, SQL Injection, SSH-Bruteforce",0.528089
9,CIC_V2,host_pair,456.0,0.233487,0.491116,DoS attacks-Hulk,"DoS attacks-GoldenEye, DoS attacks-SlowHTTPTest, DoS attacks-Slowloris, FTP-BruteForce, SQL Injection, SSH-Bruteforce",0.511303
